In [ ]:
pip install nltk tensorflow numpy

In [ ]:
import nltk
import numpy as np
import json
import random
from nltk.stem import LancasterStemmer
stemmer = LancasterStemmer()
nltk.download('punkt')

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
intents = {
  "intents": [
    {
      "tag": "greeting",
      "patterns": ["Hi", "Hello", "Hey", "How are you", "Good day"],
      "responses": ["Hello!", "Hey there!", "Hi! How can I help you?"]
    },
    {
      "tag": "goodbye",
      "patterns": ["Bye", "See you", "Goodbye", "See you later"],
      "responses": ["Goodbye!", "See you later!", "Have a great day!"]
    },
    {
      "tag": "name",
      "patterns": ["What is your name", "Who are you", "What should I call you"],
      "responses": ["I am Aravind Bot!", "You can call me Aravind Bot!"]
    },
    {
      "tag": "age",
      "patterns": ["How old are you", "What is your age"],
      "responses": ["I am just a bot, I have no age!", "Age is just a number for me!"]
    },
    {
      "tag": "thanks",
      "patterns": ["Thanks", "Thank you", "That is helpful"],
      "responses": ["Happy to help!", "Anytime!", "You are welcome!"]
    },
    {
      "tag": "about",
      "patterns": ["What can you do", "Help me", "What do you know"],
      "responses": ["I can answer your questions!", "I am here to assist you!"]
    }
  ]
}

In [ ]:
words = []
labels = []
docs_x = []
docs_y = []

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        wrds = nltk.word_tokenize(pattern)
        words.extend(wrds)
        docs_x.append(wrds)
        docs_y.append(intent["tag"])
    if intent["tag"] not in labels:
        labels.append(intent["tag"])

words = [stemmer.stem(w.lower()) for w in words if w != "?"]
words = sorted(list(set(words)))
labels = sorted(labels)

print("Words:", words)
print("Labels:", labels)

Words: ['ag', 'ar', 'bye', 'cal', 'can', 'day', 'do', 'good', 'goodby', 'hello', 'help', 'hey', 'hi', 'how', 'i', 'is', 'know', 'lat', 'me', 'nam', 'old', 'see', 'should', 'thank', 'that', 'what', 'who', 'yo', 'you']
Labels: ['about', 'age', 'goodbye', 'greeting', 'name', 'thanks']


In [ ]:
training = []
output = []
out_empty = [0] * len(labels)

for x, doc in enumerate(docs_x):
    bag = []
    wrds = [stemmer.stem(w.lower()) for w in doc]
    for w in words:
        if w in wrds:
            bag.append(1)
        else:
            bag.append(0)
    output_row = out_empty[:]
    output_row[labels.index(docs_y[x])] = 1
    training.append(bag)
    output.append(output_row)

training = np.array(training)
output = np.array(output)
print("Training data ready!")

Training data ready!


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

### Re-preparing Training Data

Due to previous executions, the `training` and `output` variables might have been reset. I will re-prepare them here to ensure the model has the correct data for training.

In [ ]:
training = []
output = []
out_empty = [0] * len(labels)

for x, doc in enumerate(docs_x):
    bag = []
    wrds = [stemmer.stem(w.lower()) for w in doc]
    for w in words:
        if w in wrds:
            bag.append(1)
        else:
            bag.append(0)
    output_row = out_empty[:]
    output_row[labels.index(docs_y[x])] = 1
    training.append(bag)
    output.append(output_row)

training = np.array(training)
output = np.array(output)
print("Training data re-prepared!")

Training data re-prepared!


### Neural Network Model Architecture

I will create a sequential model with three densely connected layers.

*   The first layer will have 128 neurons and use the ReLU activation function.
*   A dropout layer will be added to prevent overfitting.
*   The second hidden layer will have 64 neurons, also with ReLU activation.
*   Another dropout layer will be included.
*   The output layer will have `len(labels)` neurons (equal to the number of intent tags) and use a softmax activation function to output probabilities for each class.

In [ ]:
# Build the neural network model
model = Sequential()
model.add(Dense(128, input_shape=(len(training[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(labels), activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,486 (48.77 KB)

 Trainable params: 12,486 (48.77 KB)

 Non-trainable params: 0 (0.00 B)

### Compile and Train the Model

I will compile the model using the Adam optimizer, categorical cross-entropy loss, and accuracy as the metric. Then, the model will be trained using the `training` and `output` data for 200 epochs.

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(training, output, epochs=200, batch_size=5, verbose=1)

print("Model training complete!")

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2000 - loss: 1.8318   
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3500 - loss: 1.7342 
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1000 - loss: 1.7785 
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3500 - loss: 1.6937 
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3000 - loss: 1.7323 
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3000 - loss: 1.6861
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3500 - loss: 1.7580
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4000 - loss: 1.6674 
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4000 - loss: 1.6434 
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3000 - loss: 1.6640 
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5000 - loss: 1.5548
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3000 - l

### Preprocessing Input Sentence

This function `bag_of_words` will convert a user's input sentence into a bag-of-words array, which is the format expected by our neural network model. It will stem each word in the sentence and check if it exists in our vocabulary (`words`).

In [ ]:
def bag_of_words(s, words):
    bag = [0 for _ in range(len(words))]

    s_words = nltk.word_tokenize(s)
    s_words = [stemmer.stem(word.lower()) for word in s_words]

    for se in s_words:
        for i, w in enumerate(words):
            if w == se:
                bag[i] = 1
    return np.array(bag)

### Chatbot Response Function

This `chat` function will take a user's input, convert it to a bag-of-words, use the trained model to predict the intent, and then select a random response from the corresponding intent tag defined in our `intents` data.

In [ ]:
def chat():
    print("Start talking with the bot (type quit to stop)!")
    while True:
        inp = input("You: ")
        if inp.lower() == "quit":
            break

        results = model.predict(np.array([bag_of_words(inp, words)]))[0]
        results_index = np.argmax(results)
        tag = labels[results_index]

        if results[results_index] > 0.7: # Confidence threshold
            for tg in intents["intents"]:
                if tg['tag'] == tag:
                    responses = tg['responses']
            print(random.choice(responses))
        else:
            print("I didn't understand that, please try again.")

### Start Chatting!

Now you can run the `chat()` function to interact with your trained chatbot.

In [ ]:
chat()

Start talking with the bot (type quit to stop)!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Hey there!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
I am Aravind Bot!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
I can answer your questions!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Age is just a number for me!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
I didn't understand that, please try again.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Happy to help!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
I didn't understand that, please try again.
